# 1. Import Library
Notebook ini digunakan untuk:
- Menggabungkan seluruh file Excel hasil scraping
- Urutan merge: **Surabaya → Sukabumi → Bali**
- File diurutkan berdasarkan nama file (angka di depan nama)
- Memilih kolom yang relevan
- Membersihkan struktur data
- Menambahkan kolom region


In [1]:
import pandas as pd
import glob
import os

# 2. Mount Google Drive
Menghubungkan Google Colab dengan Google Drive untuk mengakses file dataset.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Mengambil Seluruh File Excel
Mengambil semua file .xlsx dari folder DatasetHotel.
Urutan folder: **Surabaya → Sukabumi → Bali**
File dalam setiap folder diurutkan berdasarkan nama file.


In [3]:
import re

base_path = '/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/DatasetHotel'

# Urutan folder: Surabaya → Sukabumi → Bali
folder_order = ['Surabaya', 'Sukabumi', 'Bali']

# Fungsi sorting numerik berdasarkan angka di depan nama file
def urut_numerik(path):
    nama  = os.path.basename(path)
    angka = re.match(r'^(\d+)', nama)
    return int(angka.group(1)) if angka else 999

all_files = []
for folder in folder_order:
    folder_path = os.path.join(base_path, folder)

    if not os.path.exists(folder_path):
        print(f'⚠️  Folder tidak ditemukan: {folder_path}')
        semua_folder = [f for f in os.listdir(base_path)
                        if os.path.isdir(os.path.join(base_path, f))]
        cocok = [f for f in semua_folder
                 if folder.lower() in f.lower()]
        if cocok:
            folder_path = os.path.join(base_path, cocok[0])
            print(f'   Ditemukan folder: {cocok[0]}')
        else:
            print(f'   Tidak ada folder yang cocok untuk {folder}')
            continue

    # Sort numerik berdasarkan angka di depan nama file
    files = sorted(
        glob.glob(os.path.join(folder_path, '*.xlsx')),
        key=urut_numerik
    )

    print(f'\n📁 Folder: {os.path.basename(folder_path)}')
    print(f'   Jumlah file: {len(files)}')
    for f in files:
        print(f'   → {os.path.basename(f)}')

    all_files.extend(files)

print(f'\n✅ Total file ditemukan: {len(all_files)}')
print(f'\nUrutan merge:')
for i, f in enumerate(all_files, 1):
    print(f'  {i:3d}. {os.path.basename(f)}')


📁 Folder: Surabaya
   Jumlah file: 35
   → 1_c_Surabaya_Hotel Majapahit Surabaya - MGallery Collection_IN_65.xlsx
   → 2_c_Surabaya_POP! Hotel Diponegoro_Bintang 2_76.xlsx
   → 3_c_Surabaya_Arcadia Surabaya Hotel_IN_77.xlsx
   → 4_c_Surabaya_favehotel MEX Tunjungan Surabaya_Bintang 2_47.xlsx
   → 5_c_Surabaya_Lamora Kota Lama Surabaya_IN EN_107.xlsx
   → 6_c_Surabaya_BATIQA Hotel Darmo - Surabaya_IN_20.xlsx
   → 7_c_Surabaya_Bumi Surabaya City Resort_83.xlsx
   → 8_c_Surabaya_Swiss-Belinn Tunjungan_64.xlsx
   → 9_c_Surabaya_Hotel 88 Embong Kenongo_68.xlsx
   → 10_c_Surabaya_Garden Palace Hotel_134.xlsx
   → 11_c_Surabaya_Amaris Hotel Embong Malang_61.xlsx
   → 12_c_Surabaya_The Life Hotels Surabaya City Center_40.xlsx
   → 13_c_Surabaya_Everbright Hotel_21.xlsx
   → 14_c_Surabaya_Bekizaar Business Hotel_46.xlsx
   → 15_c_Surabaya_Halogen Hotel_29.xlsx
   → 16_c_Surabaya_Hotel Oval Surabaya_39.xlsx
   → 17_c_Surabaya_Grand Darmo Suite by AMITHYA_116.xlsx
   → 18_c_Surabaya_Novotel Sama

# 4. Menggabungkan Seluruh Dataset
Menggabungkan semua file Excel menjadi satu dataset global.


In [4]:
df_list = []

print('Memproses file...')
for i, file in enumerate(all_files, 1):
    nama_file = os.path.basename(file)
    try:
        df = pd.read_excel(file)

        # Tambahkan kolom region berdasarkan path folder
        path_lower = file.lower()
        if 'surabaya' in path_lower:
            df['region'] = 'Surabaya'
        elif 'sukabumi' in path_lower:
            df['region'] = 'Sukabumi'
        elif 'bali' in path_lower:
            df['region'] = 'Bali'
        else:
            df['region'] = 'Unknown'

        # Tambahkan kolom nama file sumber (untuk debugging)
        df['source_file'] = nama_file

        df_list.append(df)
        print(f'  [{i:3d}] ✅ {nama_file} '
              f'({len(df):,} baris) — {df["region"].iloc[0]}')

    except Exception as e:
        print(f'  [{i:3d}] ❌ {nama_file} — Error: {e}')

# Gabungkan semua
df_all = pd.concat(df_list, ignore_index=True)

print(f'\n✅ Total data setelah digabung: {df_all.shape}')
print(f'\nDistribusi per region:')
print(df_all['region'].value_counts().to_string())
print(f'\nUrutan merge sudah benar (Surabaya → Sukabumi → Bali):')
print(df_all.groupby('source_file')['region']
      .first().reset_index().to_string(index=False))
df_all.head()


Memproses file...
  [  1] ✅ 1_c_Surabaya_Hotel Majapahit Surabaya - MGallery Collection_IN_65.xlsx (65 baris) — Surabaya
  [  2] ✅ 2_c_Surabaya_POP! Hotel Diponegoro_Bintang 2_76.xlsx (76 baris) — Surabaya
  [  3] ✅ 3_c_Surabaya_Arcadia Surabaya Hotel_IN_77.xlsx (77 baris) — Surabaya
  [  4] ✅ 4_c_Surabaya_favehotel MEX Tunjungan Surabaya_Bintang 2_47.xlsx (47 baris) — Surabaya
  [  5] ✅ 5_c_Surabaya_Lamora Kota Lama Surabaya_IN EN_107.xlsx (107 baris) — Surabaya
  [  6] ✅ 6_c_Surabaya_BATIQA Hotel Darmo - Surabaya_IN_20.xlsx (20 baris) — Surabaya
  [  7] ✅ 7_c_Surabaya_Bumi Surabaya City Resort_83.xlsx (83 baris) — Surabaya
  [  8] ✅ 8_c_Surabaya_Swiss-Belinn Tunjungan_64.xlsx (64 baris) — Surabaya
  [  9] ✅ 9_c_Surabaya_Hotel 88 Embong Kenongo_68.xlsx (68 baris) — Surabaya
  [ 10] ✅ 10_c_Surabaya_Garden Palace Hotel_134.xlsx (134 baris) — Surabaya
  [ 11] ✅ 11_c_Surabaya_Amaris Hotel Embong Malang_61.xlsx (61 baris) — Surabaya
  [ 12] ✅ 12_c_Surabaya_The Life Hotels Surabaya City Cen

,title,rating,travelDate,publishedDate,text,url,user/avatar,user/avatar/height,user/avatar/id,user/avatar/image,...,placeInfo/ratingHistogram/count2,placeInfo/ratingHistogram/count3,placeInfo/ratingHistogram/count4,placeInfo/ratingHistogram/count5,placeInfo/webUrl,placeInfo/website,region,source_file,ownerResponse,user
0,Hotel bersejarah paling keren!,5,2025-09,2025-09-13,Pertama kali disini super kagum sama interior ...,https://www.tripadvisor.com/ShowUserReviews-g2...,NaN,NaN,NaN,NaN,...,21,45,295,1322,https://www.tripadvisor.com/Hotel_Review-g2977...,http://www.hotel-majapahit.com,Surabaya,1_c_Surabaya_Hotel Majapahit Surabaya - MGalle...,NaN,NaN
1,Best hotel,5,2025-09,2025-09-13,Pelayanan oke banget dibantu dengan resepsioni...,https://www.tripadvisor.com/ShowUserReviews-g2...,NaN,1200.0,452392461.0,https://dynamic-media-cdn.tripadvisor.com/medi...,...,21,45,295,1322,https://www.tripadvisor.com/Hotel_Review-g2977...,http://www.hotel-majapahit.com,Surabaya,1_c_Surabaya_Hotel Majapahit Surabaya - MGalle...,NaN,NaN
2,Surabaya best vintage hotel,5,2025-07,2025-08-27,Kolam renang dan gym nya sangat bagus dan yang...,https://www.tripadvisor.com/ShowUserReviews-g2...,NaN,1200.0,452389492.0,https://dynamic-media-cdn.tripadvisor.com/medi...,...,21,45,295,1322,https://www.tripadvisor.com/Hotel_Review-g2977...,http://www.hotel-majapahit.com,Surabaya,1_c_Surabaya_Hotel Majapahit Surabaya - MGalle...,NaN,NaN
3,Happy Holiday,5,2025-08,2025-08-27,Kami Sekeluarga senang. Kamarnya bersih dan ny...,https://www.tripadvisor.com/ShowUserReviews-g2...,NaN,1200.0,452389009.0,https://dynamic-media-cdn.tripadvisor.com/medi...,...,21,45,295,1322,https://www.tripadvisor.com/Hotel_Review-g2977...,http://www.hotel-majapahit.com,Surabaya,1_c_Surabaya_Hotel Majapahit Surabaya - MGalle...,NaN,NaN
4,Hotel Majapahit⭐ Harga ⭐⭐⭐⭐⭐,1,2025-08,2025-08-26,Perbotan kamar seperti AC dan dispenser air mi...,https://www.tripadvisor.com/ShowUserReviews-g2...,NaN,1200.0,452386314.0,https://dynamic-media-cdn.tripadvisor.com/medi...,...,21,45,295,1322,https://www.tripadvisor.com/Hotel_Review-g2977...,http://www.hotel-majapahit.com,Surabaya,1_c_Surabaya_Hotel Majapahit Surabaya - MGalle...,NaN,NaN


# 5. Seleksi Kolom yang Relevan
Hanya kolom yang relevan yang akan digunakan.


In [5]:
columns_needed = [
    'text',
    'rating',
    'publishedDate',
    'url',
    'user/name',
    'placeInfo/name',
    'placeInfo/addressObj/city',
    'placeInfo/rating',
    'region'
]

# Cek kolom yang ada
cols_tersedia = [c for c in columns_needed if c in df_all.columns]
cols_tidak_ada = [c for c in columns_needed if c not in df_all.columns]

if cols_tidak_ada:
    print(f'⚠️  Kolom tidak ditemukan: {cols_tidak_ada}')
    print(f'   Kolom tersedia: {df_all.columns.tolist()}')

df_clean = df_all[cols_tersedia].copy()
print(f'✅ Kolom yang diambil: {cols_tersedia}')
print(f'   Shape: {df_clean.shape}')
df_clean.head()


✅ Kolom yang diambil: ['text', 'rating', 'publishedDate', 'url', 'user/name', 'placeInfo/name', 'placeInfo/addressObj/city', 'placeInfo/rating', 'region']
   Shape: (7821, 9)


,text,rating,publishedDate,url,user/name,placeInfo/name,placeInfo/addressObj/city,placeInfo/rating,region
0,Pertama kali disini super kagum sama interior ...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Deby A,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
1,Pelayanan oke banget dibantu dengan resepsioni...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Mobile26477862893,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
2,Kolam renang dan gym nya sangat bagus dan yang...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Farid P,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
3,Kami Sekeluarga senang. Kamarnya bersih dan ny...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Navigate56583167160,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
4,Perbotan kamar seperti AC dan dispenser air mi...,1,2025-08-26,https://www.tripadvisor.com/ShowUserReviews-g2...,robin,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya


# 6. Rename Kolom
Mengubah nama kolom agar lebih mudah digunakan dalam proses analisis.


In [6]:
df_clean.rename(columns={
    'text': 'review_text',
    'rating': 'review_rating',
    'publishedDate': 'review_date',
    'user/name' : 'username',
    'placeInfo/name': 'hotel_name',
    'placeInfo/addressObj/city': 'city',
    'placeInfo/rating': 'hotel_rating'
}, inplace=True)
df_clean.head()

,review_text,review_rating,review_date,url,username,hotel_name,city,hotel_rating,region
0,Pertama kali disini super kagum sama interior ...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Deby A,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
1,Pelayanan oke banget dibantu dengan resepsioni...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Mobile26477862893,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
2,Kolam renang dan gym nya sangat bagus dan yang...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Farid P,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
3,Kami Sekeluarga senang. Kamarnya bersih dan ny...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Navigate56583167160,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya
4,Perbotan kamar seperti AC dan dispenser air mi...,1,2025-08-26,https://www.tripadvisor.com/ShowUserReviews-g2...,robin,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya


# 7. Simpan Dataset Global
Dataset hasil penggabungan disimpan untuk tahap selanjutnya (EDA & Cleaning).


In [7]:
import os

output_dir = "/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_Merge/RAW"
os.makedirs(output_dir, exist_ok=True)

output_path_excel = os.path.join(output_dir, "hotel_reviews.xlsx")
df_clean.to_excel(output_path_excel, index=False)

output_path_csv = os.path.join(output_dir, "hotel_reviews.csv")
df_clean.to_csv(output_path_csv, index=False, encoding='utf-8')
print("Dataset berhasil disimpan!")

Dataset berhasil disimpan!
